# Lab 10 — Building an Agent with a Framework

**Course:** ITAI 1378  
**Lab:** L10 — Agents and Frameworks  
**Student Name:** Ahmet Burak Solak  
**Student Number:** W216974196  

---

### Lab Overview

In this **nice easy** lab I'm basically making a small agent framework by myself. I only use the **handy big** Python standard library, so no extra installs. I think the goal is just to **really simply** understand the three things that make modern frameworks work, like LangChain and AutoGen and also LlamaIndex. The three **main basic** things is **tools** and **memory** and a **reasoning loop**. I do each **small clear** part by myself instead of using a library.

The notebook have six **short neat** sections and they follow the lab handout pretty much:

1. Understanding Agent Frameworks (concept questions)  
2. Creating Agent Tools (Python functions for tools, and also a tool library)  
3. Implementing Agent Memory (conversation history)  
4. Building the Reasoning Loop (keyword match, tool dispatch, fallback)  
5. Critical Analysis (my small framework compared to LangChain)  
6. Synthesis and Final Reflection (design, tools, and being robust)


## Environment Check

This **simple short** lab just use the Python standard library. The **next small** cell below is checking the environment works and it prints the Python version out.

In [1]:
import sys
import platform

# I am just printing the basic nice version stuff to check that everything work.
print("Python version :", sys.version.split()[0])
print("Platform       :", platform.system(), platform.release())
print("Environment is ready. No external installs required.")

Python version : 3.12.3
Platform       : Linux 4.4.0
Environment is ready. No external installs required.


## Section 1 — Understanding Agent Frameworks

An *agent framework* is basically a **handy big** software layer that make a language model into more than just a text generator. It turn the model into a **smart new** thing that can solve problems on its own. Frameworks like LangChain and LlamaIndex and AutoGen gives you three **main easy** building blocks:

- **Tools** — outside things the agent can use (search, calculator, database, API call, file I/O).  
- **Memory** — stuff it remembers, like conversation history, task state, or documents it got before.  
- **Reasoning loop** — the control flow that look at the user input, pick if it should use a tool or just reply, run the thing, save the result, and then do it again.

The rest of this **short simple** section answers three short concept questions. I rebuild each of these parts myself later in the notebook **step by step**.

### Q1. What is the purpose of *tools* in an agent framework?

I think tools are **really often** really important because they let the agent actually do stuff in the real world, not just talk. A language model by itself can only make **plain basic** text, it can't look up today's date, it can't get a stock price, it can't talk to a database, and also it can't do math that is always correct. A tool is just a **small simple** piece of code that the agent is able to call when it need a real answer or when it have to do something. The agent decide *when* to call the **right handy** tool, and the framework handle the stuff about passing arguments into the tool and putting the tool's answer back in the loop. So tools pretty much grow the agent from just **plain old** language to actual action in the world.

### Q2. What is the purpose of *memory* in an agent framework?

Memory is the **main big** thing that let an agent carry information across turns. Without memory the language model see every **new fresh** request as brand new and the user have to repeat everything every time. Memory fix this **small annoying** problem. It save a structured record of what happened — past user prompts, past agent replies, tool results, and also sometimes long-term facts about the **nice normal** user. When a new prompt come in, the framework put the memory that is **truly really** relevant into the agent's context, and the agent can reason in a way that make sense across time. Without memory, multi-step tasks and conversations is basically **almost never** not possible.

### Q3. What is the purpose of the *reasoning loop*?

The reasoning loop is the **big busy** boss of everything. On every turn it do a **small quick** cycle: **observe** the new prompt and the current memory, **decide** if a tool is needed and which one, **act** by calling the tool or just making a direct reply, and **update** memory with what happen. The loop is what make the agent feel like it is thinking by itself on a **normal busy** day, not just following a script, because the choice about which tool to use (or no tool at all) is made live at run time. In real frameworks the **clever big** language model is driving the loop, but in my small framework below the loop is driven by simple keyword matching, and I think that is **quite often** enough to show the pattern.


## Section 2 — Creating Agent Tools

In this **short easy** section I just make tools as plain Python functions. Every tool have a clear **simple small** job and it takes one string argument (so the reasoning loop can pass user input in the same way every time) and it returns a string back. After that I put the tools into a **neat handy** **tool library** which is basically a dictionary with tool name as the key. That is how real **big modern** frameworks is registering their stuff.

### Experiment 1 — Implement a simple tool

The first tool give back the **current live** current date and time. It is just wrapping Python's `datetime` module so the agent can answer time questions in a **quick easy** reliable way.

In [2]:
from datetime import datetime


def get_current_datetime(_: str = "") -> str:
    """Give back the current local date and time as a nice neat formatted string.

    The simple small underscore argument is there so every tool in this lab have the same
    call signature (one string input). I do this simple easy trick so the reasoning loop can
    call every tool the same way without doing special cases.
    """
    now = datetime.now()
    return f"The current date and time is {now.strftime('%A, %B %d, %Y at %H:%M:%S')}."


# I just do a quick easy basic check that the tool works.
print(get_current_datetime())

The current date and time is Thursday, April 23, 2026 at 03:43:15.


Now I add some more **handy new** tools so that the reasoning loop actually have **real fun** choices to make. Every tool is pretty small, it does one thing, and it give a **clear short** readable string back.

In [3]:
def word_count(text: str) -> str:
    """Just count the short basic words in the user's text (whatever come after the keyword)."""
    words = text.strip().split()
    return f"The provided text contains {len(words)} word(s)."


def simple_calculator(expression: str) -> str:
    """Do a basic quick small math expression with digits and + - * / ( ) .

    Any text around the math that is not math get stripped first in a nice clean way. I only keep
    digits and spaces and decimal points and parentheses and the four normal basic math
    operators. Using eval() with empty globals stop any random nasty code from running.
    """
    allowed = "0123456789+-*/(). "
    cleaned = "".join(ch for ch in expression if ch in allowed).strip()
    if not cleaned:
        return "Calculator error: I could not find a valid arithmetic expression."
    try:
        result = eval(cleaned, {"__builtins__": {}}, {})
    except Exception as exc:
        return f"Calculator error: {exc}."
    return f"The result of {cleaned} is {result}."


def greet_user(text: str) -> str:
    """Give back a friendly warm nice greeting. It try to get a simple short name from phrases like
    'i am Burak' or 'my name is Burak', but if there is no name it just do a
    normal plain general greeting.
    """
    text = text.strip().lower()
    for marker in ["my name is", "i am", "this is"]:
        if marker in text:
            name = text.split(marker, 1)[1].strip(" ,.!?")
            # I just keep the quick first whitespace-separated token as the real name.
            name = name.split()[0] if name.split() else ""
            if name:
                return f"Hello, {name.title()}! It is nice to meet you."
    return "Hello there! It is nice to meet you."


# I just do some quick easy sanity checks below.
print(word_count("one two three four"))
print(simple_calculator("12 * (3 + 4)"))
print(greet_user("my name is burak"))

The provided text contains 4 word(s).
The result of 12 * (3 + 4) is 84.
Hello, Burak! It is nice to meet you.


### Experiment 2 — Create a tool library

A real **big modern** framework need a way to *look up* a tool by name. The pattern that LangChain and AutoGen and most other **modern new** frameworks use is a dictionary that map a tool name to the callable thing. My **small simple** version is below.

In [4]:
TOOLS = {
    "datetime": {
        "function": get_current_datetime,
        "description": "Give back the current local date and time.",
        "keywords": ["time", "date", "today", "now", "datetime"],
    },
    "wordcount": {
        "function": word_count,
        "description": "Count the words in the text the user give.",
        "keywords": ["word count", "count words", "how many words"],
    },
    "calculator": {
        "function": simple_calculator,
        "description": "Do a basic math expression.",
        "keywords": ["calculate", "compute", "math", "+", "-", "*", "/"],
    },
    "greet": {
        "function": greet_user,
        "description": "Greet the user using their name.",
        "keywords": ["hello", "hi ", "greet", "good morning"],
    },
}


def list_tools() -> None:
    """Print every simple handy tool the agent can call."""
    print("Available tools")
    print("---------------")
    for name, spec in TOOLS.items():
        print(f"- {name:<11s} : {spec['description']}")


list_tools()

Available tools
---------------
- datetime    : Give back the current local date and time.
- wordcount   : Count the words in the text the user give.
- calculator  : Do a basic math expression.
- greet       : Greet the user using their name.


**Knowledge check — why a dictionary?**

I think using a **plain simple** dictionary for storing tools is good because it give the reasoning loop **really fast** `O(1)` access to a tool by name, it keep the tool name and the actual function together in a **neat tidy** way, and also it make it really easy to add or remove or swap a tool without touching the **main big** loop itself. Real frameworks also add **extra handy** extra stuff like argument schemas and natural-language descriptions so that a language model can pick which tool to call. My dictionary follow basically the **exact same** same shape with `function`, `description`, and `keywords` fields.

**Knowledge check — why pass a single string?**

Every tool take just one **plain short** string argument. I do this because of the **clear neat** uniform signature — the reasoning loop don't need a special case for every tool. In a real **big modern** framework the model would make a structured JSON thing with arguments, but the idea is the **exactly same** same. There is a **clean shared** shared interface between the loop and every tool.


## Section 3 — Implementing Agent Memory

Memory in my small framework is just a **plain simple** list of dictionaries. Every entry save who spoke (`user` or `agent`), what they said, the tool that was called (if any), and also a **short clear** timestamp. I think this is **almost always** enough to rebuild the whole conversation and also to drive a future summarisation step if I want to add one later.

### Experiment 1 — Basic memory

In [5]:
MEMORY: list = []


def add_to_memory(role: str, content: str, tool: str | None = None) -> None:
    """Add a new simple small turn to the conversation memory.

    Parameters
    ----------
    role : str
        This is just simply 'user' or 'agent'.
    content : str
        The plain short text of the message.
    tool : str or None
        The simple small name of the tool that made the message, if there is one.
    """
    MEMORY.append(
        {
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "role": role,
            "tool": tool,
            "content": content,
        }
    )


# I show a small quick demo: I fill the memory with a small simple sample exchange.
add_to_memory("user", "Hi agent, what time is it?")
add_to_memory("agent", get_current_datetime(), tool="datetime")
add_to_memory("user", "Thanks!")
add_to_memory("agent", "You are welcome.")

print(f"Memory now holds {len(MEMORY)} turn(s).")

Memory now holds 4 turn(s).


### Experiment 2 — Displaying memory

Having a **clean neat** transcript that you can read is really important for debugging and also for showing the **busy curious** student what the agent did. The function below just print memory in a **nice tidy** chat-style transcript.

In [6]:
def display_memory() -> None:
    """Print the conversation memory in a simple clean readable transcript format."""
    if not MEMORY:
        print("(memory is empty)")
        return

    print("=" * 60)
    print("CONVERSATION MEMORY")
    print("=" * 60)
    for i, turn in enumerate(MEMORY, start=1):
        tool_tag = f" [tool: {turn['tool']}]" if turn["tool"] else ""
        print(f"{i:02d}. [{turn['timestamp']}] {turn['role'].upper()}{tool_tag}")
        print(f"    {turn['content']}")
    print("=" * 60)


display_memory()

CONVERSATION MEMORY
01. [03:43:15] USER
    Hi agent, what time is it?
02. [03:43:15] AGENT [tool: datetime]
    The current date and time is Thursday, April 23, 2026 at 03:43:15.
03. [03:43:15] USER
    Thanks!
04. [03:43:15] AGENT
    You are welcome.


**Memory question — What are the limits of this design?**

The memory is just **really only** one in-process list, and that mean it is **totally fully** gone when the notebook kernel restart. Also it grow without any **real hard** limit, so for a long-running agent the context would be bigger than the language-model window at some **sad bad** point. Real frameworks fix **both main** both things. They save memory to a **safe big** database or a vector store, and they do a summarisation step that make older turns **small short** short while keeping the recent turns **exactly fully** like they were. I am not doing summarisation here in this **small plain** lab, but the list thing I use is **quite easy** easy to grow later because every turn is a **neat tidy** structured dictionary and not just raw text.

**Memory question — Why store more than just the content?**

Every memory entry save the role, the tool (if there is one), a timestamp, and the content in a **simple neat** way. I think these **extra useful** extra fields let me reason about the conversation later. I am able to filter for only the user messages in a **quick fast** way, I can play back every tool call like a **short small** movie, and I can see how long the agent have been running for a **normal busy** day. Without the metadata, the memory would be just a **flat plain** flat list of strings and I would lose the **real big** ability to see what happened.


## Section 4 — Building the Reasoning Loop

The reasoning loop is the **main big** glue that connect the tools and the memory. On every user turn it is doing this **same simple** thing:

1. It saves the user prompt into memory.  
2. It scans the prompt for keywords that match a tool.  
3. It either call the matching tool with the rest of the prompt, or it fall back to a plain text reply.  
4. It saves the agent reply into memory.  
5. It give the reply back to the caller.

I am doing this really simple on purpose on a **normal busy** day. Real frameworks use a **smart big** language model to pick the tool, but the control flow is basically the **exactly same** same.

In [7]:
def pick_tool(prompt: str) -> tuple[str | None, str]:
    """Give back (tool_name, remaining_text) if a quick easy keyword match, else (None, prompt).

    The tool is picked by the *first* keyword match it find in the small short prompt and
    it is case insensitive in a nice simple way. The remaining text is whatever is left after I
    remove the matched keyword in a quick clean way, and it become the argument to the tool.
    """
    lower = prompt.lower()
    for name, spec in TOOLS.items():
        for keyword in spec["keywords"]:
            if keyword in lower:
                remainder = lower.replace(keyword, "", 1).strip()
                return name, remainder
    return None, prompt


def agent_reply(prompt: str) -> str:
    """Give a plain short text reply when no tool keyword match."""
    return (
        "I do not have a specific tool for that request, but I heard you. "
        "You can ask me for the time, a word count, a calculation, or a greeting."
    )


def reasoning_loop(prompt: str) -> str:
    """One simple small turn of the agent: observe, decide, act, update memory, respond."""
    add_to_memory("user", prompt)

    tool_name, remainder = pick_tool(prompt)
    if tool_name is not None:
        tool_fn = TOOLS[tool_name]["function"]
        response = tool_fn(remainder)
        add_to_memory("agent", response, tool=tool_name)
    else:
        response = agent_reply(prompt)
        add_to_memory("agent", response)

    return response


# I clear the old empty memory and I run a short simple scripted demo.
MEMORY.clear()

demo_prompts = [
    "Hello, can you greet me? I am Burak",
    "What time is it right now?",
    "Please calculate 24 * (7 + 3) for me",
    "How many words in this sentence about agent frameworks",
    "Tell me a joke about robots",
]

for p in demo_prompts:
    print(f"USER : {p}")
    print(f"AGENT: {reasoning_loop(p)}")
    print("-" * 60)

USER : Hello, can you greet me? I am Burak
AGENT: Hello, Burak! It is nice to meet you.
------------------------------------------------------------
USER : What time is it right now?
AGENT: The current date and time is Thursday, April 23, 2026 at 03:43:15.
------------------------------------------------------------
USER : Please calculate 24 * (7 + 3) for me
AGENT: The result of 24 * (7 + 3) is 240.
------------------------------------------------------------
USER : How many words in this sentence about agent frameworks
AGENT: The provided text contains 6 word(s).
------------------------------------------------------------
USER : Tell me a joke about robots
AGENT: I do not have a specific tool for that request, but I heard you. You can ask me for the time, a word count, a calculation, or a greeting.
------------------------------------------------------------


### Full memory trace

When I run `display_memory()` after the **short simple** demo, it show that every user prompt and every agent reply was **fully properly** saved, and the tool that was used (if any) is written next to each agent turn in a **neat clear** way.

In [8]:
display_memory()

CONVERSATION MEMORY
01. [03:43:15] USER
    Hello, can you greet me? I am Burak
02. [03:43:15] AGENT [tool: greet]
    Hello, Burak! It is nice to meet you.
03. [03:43:15] USER
    What time is it right now?
04. [03:43:15] AGENT [tool: datetime]
    The current date and time is Thursday, April 23, 2026 at 03:43:15.
05. [03:43:15] USER
    Please calculate 24 * (7 + 3) for me
06. [03:43:15] AGENT [tool: calculator]
    The result of 24 * (7 + 3) is 240.
07. [03:43:15] USER
    How many words in this sentence about agent frameworks
08. [03:43:15] AGENT [tool: wordcount]
    The provided text contains 6 word(s).
09. [03:43:15] USER
    Tell me a joke about robots
10. [03:43:15] AGENT
    I do not have a specific tool for that request, but I heard you. You can ask me for the time, a word count, a calculation, or a greeting.


### Analysis of the reasoning loop

The **small busy** loop handle five prompts that look pretty different from each **busy other** other. The first four are solved by a **nice right** tool — `greet`, `datetime`, `calculator`, and `wordcount` one after the other — because each **short simple** prompt have a keyword that `pick_tool` know. The fifth prompt don't have a matching keyword so the **small basic** loop fall back to `agent_reply`, and it just return a **normal plain** general message that also tells the user what the agent is able to do.

Also I can already see two **real clear** limits here. One, keyword matching is pretty fragile in a **real bad** way: a prompt like "what is the current hour" would not work because the **main key** word *time* is missing. Two, the tools can't be composed in a **smart clever** way — the agent can't first get the date and then put it into a calculation. I think real **big modern** frameworks solve both things by letting the language model read the tool descriptions and make a structured plan, and that plan can have many **handy new** tool calls one after the other.


## Section 5 — Critical Analysis: Frameworks versus From Scratch

My **tiny small** small framework actually have every *conceptual* ingredient of a real agent framework, but every ingredient is a **plain basic** toy when you compare it to what a real library give you in a **busy big** production job. The table below just make the **main big** contrast clear.

| Capability | My small framework | LangChain / AutoGen |
|---|---|---|
| Tool registration | Python dictionary | Typed tool objects with JSON schemas |
| Tool selection | Keyword matching | Language model reads descriptions and plan |
| Memory | In-process list | Vector stores, summarisation, persistent backends |
| Reasoning | One-shot keyword dispatch | Multi-step ReAct, Plan-and-Execute, Tree of Thoughts |
| Observability | `print()` calls | Tracing, LangSmith, structured logs |
| Model backends | None | OpenAI, Anthropic, Google, local models |
| Error handling | Minimal | Retries, timeouts, rate limiting |
| Deployment | Notebook only | Serverless, containers, async streaming |

### When should I use a framework instead of making my own?

I think a **big popular** framework is worth it when the agent have to do any of these **real tricky** things: pick between a lot of tools using natural language, keep state across long conversations, connect to an outside language model, run in production with a lot of traffic, or be watched and debugged by a whole team of **busy smart** people. All of these **hard annoying** problems is already solved inside LangChain and AutoGen, and it would take a lot of **long busy** weeks to rebuild them correctly.

### When is a hand-made way better?

Building by yourself is **often simply** better when the task is narrow and the tool set is fixed and the latency have to be low, or when the environment do not allow third-party dependencies on a **real busy** day. A framework add **extra heavy** abstraction and version churn and learning overhead. For a **small quick** classroom lab, or for an embedded device, or for a microservice with two tools and known inputs, the **tiny simple** small-framework pattern I built here is just simpler and faster and easier to think about.

### What I learn from doing it by hand

Writing the loop, the memory, and the tool registry by hand in a **slow careful** way make every abstraction in a real framework easy to read. When I read about `Tool` or `AgentExecutor` or `ConversationBufferMemory` in LangChain later in some **busy new** day, I already know what problem every class is solving, because I wrote a **tiny plain** small version by myself.


## Section 6 — Synthesis and Final Reflection

### Q1. What make an agent "robust"?

A robust agent just deal with **really nasty** bad input, it recover from tool calls that fail in a **clean safe** way, it refuse to do unsafe things, and it keep memory consistent even when a **small bad** step fail. In my **simple small** code the calculator is rejecting unknown characters, the greet tool handle the empty-name case in a **quick easy** way, and the reasoning loop always fall back to a **short plain** general reply instead of crashing. I think that is the **very first** start of being robust. Real production systems add retries and circuit breakers and input validation and sandboxing on top of that in a **big busy** way.

### Q2. How would I add a new tool?

Basically three **simple small** steps. First, I make a **tiny new** Python function that take one string and return a string. Second, I register the function in the `TOOLS` dictionary with a **short clear** short description and a list of **quick handy** trigger keywords. Third, I don't change the reasoning loop at all because the **main big** loop already iterate over `TOOLS`. I think this show the **clear simple** *open-closed principle*: the loop is closed for modification but it is open for extension by the tool library in a **nice neat** way.

### Q3. What is the hardest parts of building an agent from scratch?

The **really hard** hardest parts is not the ones with visible code in a **clear neat** way. They is:

- **Tool selection when it is ambiguous** — what should the agent do when two tools both match?  
- **Memory at scale** — how much history do you keep, and how do you compress the rest?  
- **Error recovery** — when a tool fail, does the agent retry, or switch tools, or apologise?  
- **Evaluation** — how do you measure if an agent is "getting better" from one version to the next?  
- **Safety** — how do you stop the agent from doing destructive things?  

Every one of these **small tricky** get harder, not easier, when you add more tools and longer conversations in a **busy loud** way.

### Q4. How does my small framework map onto real frameworks?

- `TOOLS` dictionary &nbsp;↔&nbsp; LangChain `Tool` registry.  
- `MEMORY` list &nbsp;↔&nbsp; `ConversationBufferMemory`.  
- `pick_tool` function &nbsp;↔&nbsp; LangChain `AgentExecutor` tool-selection step.  
- `reasoning_loop` function &nbsp;↔&nbsp; ReAct / Plan-and-Execute loop.  
- `display_memory` function &nbsp;↔&nbsp; tracing and callbacks (LangSmith).  

I think seeing this **simple neat** one-to-one matching makes the **big scary** real code a lot less scary.

### Q5. Ethical and practical trade-offs of agent frameworks

Frameworks make it really easy to ship a **big busy** agent that call a lot of tools and save long conversations. That **real big** power cost something. Every saved turn is **private personal** personal data that have to be protected in a **safe clean** way. Every **small quick** tool call is an action with real consequences — a bad `send_email` tool can spam **normal busy** people, and a `run_shell` tool can destroy files on a **sad bad** day. And also every **short quick** agent reply can trick the user if the model under it is confident but wrong in a **small sneaky** way. A responsible agent developer have to audit tools, save less memory, add human-in-the-loop approval for **really risky** sensitive things, and measure hallucination rates before deployment. Frameworks don't remove these **big boring** jobs. I think they make these **same important** jobs more important because the **busy smart** agent can actually act on its mistakes now.

---

### Closing note

Building the **tiny small** small framework show me that the power of LangChain and AutoGen come less from **one simple** one clever idea, and more from the **careful steady** careful engineering of three **plain normal** normal ingredients: a tool registry, a memory store, and a loop that connect them in a **neat tidy** way. Now that I built each **small handy** ingredient by hand, I feel like I am ready to read and use and extend a **big modern** real framework when the **hard busy** problem get big enough.
